# Дообучение Octo в LIBERO с разными входами (глава 3, шаг 2)

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/chapter3/octo/notebooks/libero_octo_finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Берём готовую **Octo-Small 1.5** (`rail-berkeley/octo-small-1.5`, LIBERO не видела) и дообучаем её
на **тех же 432 демонстрациях LIBERO-Spatial**, на которых учились OpenVLA и OpenVLA-OFT, — несколько раз,
меняя только набор входов. Потом каждую модель проверяем на тех же 30 попытках, что в главах 1–2.

| Вариант | Входы | Вопрос |
|---|---|---|
| `cam` | внешняя камера + команда | как OpenVLA в главе 1 |
| `cam_proprio` | + датчики (положение захвата и пальцев) | помогают ли датчики — или мешают, как пишут авторы Octo? |
| `cam_wrist` | + камера на запястье | что даёт камера на запястье? |
| `cam_wrist_proprio` | всё вместе | как у OpenVLA-OFT в главе 2 |

В главе 2 этого сделать было нельзя: OFT не дообучить на бесплатной видеокарте, а готовых моделей
с разными входами нет. Octo в 250 раз меньше, поэтому дообучение помещается в T4.

**Важно:** веса сохраняются по ходу обучения. Чтобы они пережили обрыв сессии, можно подключить
Google Диск (блок 3, `USE_DRIVE = True`).

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -g | head -2

## 2. Установка (~10 мин) — то же окружение, что в шаге 1

In [ ]:
BRANCH = "chapter3/octo"   # после слияния в main поменять на "main"
PY = "/content/venv-octo/bin/python"
REPO_DIR = "/content/vla-manipulation-research"
JAX_LINKS = "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!cd vla-manipulation-research && git fetch -q origin && git reset -q --hard origin/{BRANCH} && git log --oneline -1
!test -d LIBERO || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
!sed -i 's/init_states = torch.load(init_states_path)/init_states = torch.load(init_states_path, weights_only=False)/' LIBERO/libero/libero/benchmark/__init__.py
!test -d octo || git clone -q --depth 1 https://github.com/octo-models/octo.git

!pip install -q uv
!test -d /content/venv-octo || uv venv -q --python 3.10 /content/venv-octo
!uv pip install -q --python {PY} -r {REPO_DIR}/notebooks/requirements-libero-octo.txt
!uv pip install -q --python {PY} --no-deps -e /content/octo "bddl==1.0.1"
!uv pip install -q --python {PY} --index-url https://download.pytorch.org/whl/cpu "torch==2.2.0"
!uv pip install -q --python {PY} -f {JAX_LINKS} "jax[cuda12_pip]==0.4.20" -r {REPO_DIR}/notebooks/requirements-jax-cuda.txt
!{PY} -c "import sys, jax, flax, tensorflow as tf, robosuite, octo; print('Python', sys.version.split()[0], '| jax', jax.__version__, jax.devices(), '| flax', flax.__version__, '| tf', tf.__version__)"

## 3. Где хранить веса

С Диском веса переживут обрыв сессии; без него — лежат в `/content/ckpt` до конца сессии.

In [ ]:
USE_DRIVE = False   # True — веса на Google Диске (попросит разрешение на доступ к Диску)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT = "/content/drive/MyDrive/octo_libero_ckpt"
else:
    CKPT = "/content/ckpt"
!mkdir -p {CKPT}
print("веса:", CKPT)

## 4. Демонстрации LIBERO-Spatial (~2 ГБ, ~2 мин)

Те же 432 демонстрации, что у OpenVLA и OpenVLA-OFT: `openvla/modified_libero_rlds`, папка
`libero_spatial_no_noops` (кадры внешней камеры и камеры на запястье 256×256, датчики, действия, команда).

In [ ]:
DATA = "/content/libero_rlds"
import os
# при обрыве связи скачивание повторяется и продолжается с места обрыва
for attempt in range(5):
    !{PY} -c "from huggingface_hub import snapshot_download; snapshot_download('openvla/modified_libero_rlds', repo_type='dataset', allow_patterns=['libero_spatial_no_noops/*'], local_dir='{DATA}', max_workers=4)"
    folder = f"{DATA}/libero_spatial_no_noops/1.0.0"
    n = len([f for f in os.listdir(folder) if "tfrecord" in f]) if os.path.isdir(folder) else 0
    print(f"файлов данных: {n} из 16")
    if n == 16:
        break
!du -sh {DATA}/libero_spatial_no_noops

## 5. Пробный запуск: 200 шагов

Проверяем, что обучение идёт и влезает в память, и меряем скорость — по ней выбираем число шагов
в блоке 6. Самый тяжёлый вариант — со всеми входами.

In [ ]:
FT = f"{REPO_DIR}/scripts/octo_finetune_libero.py"
BATCH = 32   # 64 со всеми входами не помещается в память T4 (нужно ~12 ГБ на промежуточные вычисления)
!{PY} {FT} --data-dir {DATA} --wrist --proprio --steps 200 --batch-size {BATCH} --warmup 50 --save-every 100000 --save-dir /content/ckpt_probe

## 6. Дообучение и проверка четырёх вариантов

Каждый вариант сначала учится (`STEPS` шагов — по скорости из блока 5), затем сразу проверяется на тех же
30 попытках LIBERO-Spatial, что в главах 1–2. Модель получает на проверке только те входы, на которых её
учили, — с живыми показаниями из симулятора. Если сессия оборвётся, готовые варианты останутся в
`results`; запустите блок снова, убрав их из `RUN`.

Для ориентира: автор готовой модели из шага 1 учил 60 000 шагов пачками по 256. Нам столько не успеть,
поэтому все варианты учатся одинаково недолго, и сравнивать их корректно только между собой.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

STEPS = 3000   # ~1 шаг/с на T4 при пачке 32: ~50 мин на вариант
# Старт от исходной Octo-Small 1.5 (LIBERO не видела): за 3000 шагов вариант cam не освоил задачи (0/30)
VARIANTS = {"cam": "", "cam_proprio": "--proprio", "cam_wrist": "--wrist", "cam_wrist_proprio": "--wrist --proprio",
            # старт от Octo из шага 1 (уже 60 000 шагов на LIBERO, только камера, 22/30): обе модели получают
            # одинаковые 3000 шагов, разница — только датчики на входе
            "lib_cam": "--pretrained octo_libero --lr 1e-4 --warmup 300",
            "lib_cam_proprio": "--pretrained octo_libero --lr 1e-4 --warmup 300 --proprio",
            # упрощённый FuSe: + дополнительное задание «описать, что показывают датчики» (scripts/octo_aux_head.py)
            "lib_cam_proprio_fuse": "--pretrained octo_libero --lr 1e-4 --warmup 300 --proprio --fuse-aux 0.5"}
RUN = ["lib_cam", "lib_cam_proprio", "lib_cam_proprio_fuse"]

EVAL = f"{REPO_DIR}/scripts/libero_octo_eval.py"
OUT = "/content/results/octo_finetune_libero_spatial"
TASKS = " ".join(str(t) for t in range(10))
# служебные предупреждения TensorFlow, robosuite и т.п. не показываем
NOISE = "Unable to register|cpu_feature_guard|rebuild TensorFlow|TensorRT|robosuite|datasets path|Gym|gymnasium|tokenizers|TOKENIZERS|fork|^$"
for name in RUN:
    print(f"===== {name}: обучение =====")
    !{PY} {FT} --data-dir {DATA} {VARIANTS[name]} --steps {STEPS} --batch-size {BATCH} --save-dir {CKPT}/{name} 2>&1 | grep -Ev "{NOISE}"
    print(f"===== {name}: проверка в LIBERO =====")
    !{PY} {EVAL} --libero-root /content/LIBERO --suite libero_spatial --model local --checkpoint {CKPT}/{name} --step {STEPS} --tasks {TASKS} --trials 3 --out {OUT}/{name} 2>&1 | grep -Ev "{NOISE}"
    if "proprio" in name:  # та же модель с «зависшими» датчиками: пользуется ли она ими?
        !{PY} {EVAL} --libero-root /content/LIBERO --suite libero_spatial --model local --checkpoint {CKPT}/{name} --step {STEPS} --tasks {TASKS} --trials 3 --freeze-proprio --out {OUT}/{name}_frozen 2>&1 | grep -Ev "{NOISE}"

## 7. Результаты

Для сравнения на тех же 30 попытках: OpenVLA (глава 1) — 23/30; OpenVLA-OFT (глава 2) — 30/30;
готовая Octo из шага 1 (60 000 шагов обучения, только внешняя камера).

In [ ]:
import json
import numpy as np
import pandas as pd

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * max(c - h, 0), 100 * min(c + h, 1)

rows = []
for name in RUN:
    path = f"{OUT}/{name}/summary.csv"
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    meta = json.load(open(f"{OUT}/{name}/meta.json"))
    log = json.load(open(f"{CKPT}/{name}/train_log.json"))
    k, n = int(df.success.sum()), len(df)
    lo, hi = wilson(k, n)
    rows.append({"вариант": name, "входы": ", ".join(meta["inputs"]), "успех": f"{k}/{n} ({100 * k / n:.0f}%)",
                 "95% ДИ": f"[{lo:.0f}%, {hi:.0f}%]", "ошибка в конце обучения": round(log[-1]["loss"], 4),
                 "шагов в успешной попытке": round(df[df.success].steps.mean(), 0)})
pd.DataFrame(rows)

## 8. Сохранение результатов

In [ ]:
!cd /content/results && zip -qr /content/results_octo_finetune.zip octo_finetune_libero_spatial && ls -lh /content/results_octo_finetune.zip
!cd {CKPT} && zip -qr /content/results_octo_finetune.zip */train_log.json */run.json
from google.colab import files
files.download("/content/results_octo_finetune.zip")